In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

BASE_DIR = Path.cwd().parent

RAW_PATH = (
    BASE_DIR
    / "data"
    / "raw"
    / "WA_Fn-UseC_-Telco-Customer-Churn.csv"
)

df = pd.read_csv(RAW_PATH)

print("Dataset shape:", df.shape)

df.head()

Dataset shape: (7043, 21)


,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


In [2]:
# Total charges

df["TotalCharges"] = pd.to_numeric(
    df["TotalCharges"],
    errors="coerce"
)

df["TotalCharges"] = df["TotalCharges"].fillna(0)

print(df["TotalCharges"].dtype)
print("Missing values:", df["TotalCharges"].isnull().sum())

float64
Missing values: 0


In [3]:
# Feature 1 — Tenure Group

df["TenureGroup"] = pd.cut(
    df["tenure"],
    bins=[-1, 12, 24, 48, 72],
    labels=[
        "New",
        "Early",
        "Mid",
        "Long-term"
    ]
)

df["TenureGroup"].value_counts()

TenureGroup
Long-term    2239
New          2186
Mid          1594
Early        1024
Name: count, dtype: int64

In [4]:
# Feature 2 — New Customer Indicator

df["IsNewCustomer"] = (
    df["tenure"] <= 12
).astype(int)

df["IsNewCustomer"].value_counts()

IsNewCustomer
0    4857
1    2186
Name: count, dtype: int64

In [5]:
df

,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn,TenureGroup,IsNewCustomer
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No,New,1
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,No,No,One year,No,Mailed check,56.95,1889.50,No,Mid,0
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes,New,1
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No,Mid,0
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes,New,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7038,6840-RESVB,Male,0,Yes,Yes,24,Yes,Yes,DSL,Yes,...,Yes,Yes,One year,Yes,Mailed check,84.80,1990.50,No,Early,0
7039,2234-XADUH,Female,0,Yes,Yes,72,Yes,Yes,Fiber optic,No,...,Yes,Yes,One year,Yes,Credit card (automatic),103.20,7362.90,No,Long-term,0
7040,4801-JZAZL,Female,0,Yes,Yes,11,No,No phone service,DSL,Yes,...,No,No,Month-to-month,Yes,Electronic check,29.60,346.45,No,New,1
7041,8361-LTMKD,Male,1,Yes,No,4,Yes,Yes,Fiber optic,No,...,No,No,Month-to-month,Yes,Mailed check,74.40,306.60,Yes,New,1


In [6]:
# Feature 3 — Long-Term Customer Indicator

df["IsLongTermCustomer"] = (
    df["tenure"] >= 48
).astype(int)

df["IsLongTermCustomer"].value_counts()

IsLongTermCustomer
0    4740
1    2303
Name: count, dtype: int64

In [7]:
# Feature 4 — Number of Services

service_columns = [
    "PhoneService",
    "MultipleLines",
    "InternetService",
    "OnlineSecurity",
    "OnlineBackup",
    "DeviceProtection",
    "TechSupport",
    "StreamingTV",
    "StreamingMovies"
]

df["NumberOfServices"] = 0

for column in service_columns:
    df["NumberOfServices"] += (
        ~df[column].isin(["No", "No internet service"])
    ).astype(int)

df["NumberOfServices"].describe()

count    7043.000000
mean        4.243078
std         2.307141
min         1.000000
25%         2.000000
50%         4.000000
75%         6.000000
max         9.000000
Name: NumberOfServices, dtype: float64

In [8]:
df[
    [
        "customerID",
        "NumberOfServices"
    ]
].head(10)

,customerID,NumberOfServices
0,7590-VHVEG,3
1,5575-GNVDE,4
2,3668-QPYBK,4
3,7795-CFOCW,5
4,9237-HQITU,2
5,9305-CDSKC,6
6,1452-KIOVK,5
7,6713-OKOMC,3
8,7892-POOKP,7
9,6388-TABGU,4


In [9]:
# Feature 5 — Optional Service Count

optional_service_columns = [
    "OnlineSecurity",
    "OnlineBackup",
    "DeviceProtection",
    "TechSupport",
    "StreamingTV",
    "StreamingMovies"
]

df["OptionalServiceCount"] = 0

for column in optional_service_columns:
    df["OptionalServiceCount"] += (
        df[column] == "Yes"
    ).astype(int)

df["OptionalServiceCount"].value_counts().sort_index()

OptionalServiceCount
0    2219
1     966
2    1033
3    1118
4     852
5     571
6     284
Name: count, dtype: int64

In [10]:
# Feature 6 — Internet Service Indicator

df["HasInternetService"] = (
    df["InternetService"] != "No"
).astype(int)

df["HasInternetService"].value_counts()

HasInternetService
1    5517
0    1526
Name: count, dtype: int64

In [11]:
# Feature 7 — Fiber Optic Indicator

df["IsFiberCustomer"] = (
    df["InternetService"] == "Fiber optic"
).astype(int)

df["IsFiberCustomer"].value_counts()

IsFiberCustomer
0    3947
1    3096
Name: count, dtype: int64

In [12]:
# Feature 8 — Monthly Charge Group

monthly_charge_q1 = df["MonthlyCharges"].quantile(0.25)
monthly_charge_q2 = df["MonthlyCharges"].quantile(0.50)
monthly_charge_q3 = df["MonthlyCharges"].quantile(0.75)

print("Q1:", monthly_charge_q1)
print("Q2:", monthly_charge_q2)
print("Q3:", monthly_charge_q3)

Q1: 35.5
Q2: 70.35
Q3: 89.85


In [13]:
df["MonthlyChargeGroup"] = pd.cut(
    df["MonthlyCharges"],
    bins=[
        -np.inf,
        monthly_charge_q1,
        monthly_charge_q2,
        monthly_charge_q3,
        np.inf
    ],
    labels=[
        "Low",
        "Medium",
        "High",
        "Very High"
    ]
)

df["MonthlyChargeGroup"].value_counts()

MonthlyChargeGroup
Medium       1766
Low          1762
Very High    1758
High         1757
Name: count, dtype: int64

In [14]:
# Feature 9 — Total Charge Group

total_charge_q1 = df["TotalCharges"].quantile(0.25)
total_charge_q2 = df["TotalCharges"].quantile(0.50)
total_charge_q3 = df["TotalCharges"].quantile(0.75)

df["TotalChargeGroup"] = pd.cut(
    df["TotalCharges"],
    bins=[
        -np.inf,
        total_charge_q1,
        total_charge_q2,
        total_charge_q3,
        np.inf
    ],
    labels=[
        "Low",
        "Medium",
        "High",
        "Very High"
    ]
)

df["TotalChargeGroup"].value_counts()

TotalChargeGroup
Low          1762
Very High    1761
Medium       1760
High         1760
Name: count, dtype: int64

In [15]:
# Feature 10 — Charge-to-Tenure Ratio

df["ChargePerTenure"] = np.where(
    df["tenure"] > 0,
    df["TotalCharges"] / df["tenure"],
    df["MonthlyCharges"]
)

df["ChargePerTenure"].describe()

count    7043.000000
mean       64.762906
std        30.189796
min        13.775000
25%        35.935156
50%        70.337500
75%        90.174158
max       121.400000
Name: ChargePerTenure, dtype: float64

In [16]:
# Feature 11 — High-Value Customer

high_value_threshold = df["MonthlyCharges"].quantile(0.75)

print("High-value threshold:", high_value_threshold)

df["HighValueCustomer"] = (
    df["MonthlyCharges"] >= high_value_threshold
).astype(int)

df["HighValueCustomer"].value_counts()

High-value threshold: 89.85


HighValueCustomer
0    5272
1    1771
Name: count, dtype: int64

In [17]:
# Feature 12 — Contract Risk Indicator

df["MonthToMonthContract"] = (
    df["Contract"] == "Month-to-month"
).astype(int)

df["MonthToMonthContract"].value_counts()

MonthToMonthContract
1    3875
0    3168
Name: count, dtype: int64

In [18]:
# Feature 13 — Electronic Check Indicator

df["ElectronicCheck"] = (
    df["PaymentMethod"] == "Electronic check"
).astype(int)

df["ElectronicCheck"].value_counts()

ElectronicCheck
0    4678
1    2365
Name: count, dtype: int64

In [19]:
# Feature 14 — No Tech Support Indicator    

df["NoTechSupport"] = (
    df["TechSupport"] == "No"
).astype(int)

df["NoTechSupport"].value_counts()

NoTechSupport
0    3570
1    3473
Name: count, dtype: int64

In [20]:
# Feature 15 — No Online Security Indicator

df["NoOnlineSecurity"] = (
    df["OnlineSecurity"] == "No"
).astype(int)

df["NoOnlineSecurity"].value_counts()

NoOnlineSecurity
0    3545
1    3498
Name: count, dtype: int64

In [21]:
# Feature 16 — Service Risk Indicator

df["ServiceRiskIndicator"] = (
    (
        (df["MonthToMonthContract"] == 1) &
        (df["NoOnlineSecurity"] == 1) &
        (df["NoTechSupport"] == 1)
    )
).astype(int)

df["ServiceRiskIndicator"].value_counts()

ServiceRiskIndicator
0    4878
1    2165
Name: count, dtype: int64

In [22]:
# Feature 17 — High-Risk Customer Indicator

df["HighRiskCustomer"] = (
    (
        (df["MonthToMonthContract"] == 1) &
        (df["IsNewCustomer"] == 1) &
        (df["HighValueCustomer"] == 1)
    )
).astype(int)

df["HighRiskCustomer"].value_counts()

HighRiskCustomer
0    6827
1     216
Name: count, dtype: int64

In [23]:
# Check all new features

new_features = [
    "TenureGroup",
    "IsNewCustomer",
    "IsLongTermCustomer",
    "NumberOfServices",
    "OptionalServiceCount",
    "HasInternetService",
    "IsFiberCustomer",
    "MonthlyChargeGroup",
    "TotalChargeGroup",
    "ChargePerTenure",
    "HighValueCustomer",
    "MonthToMonthContract",
    "ElectronicCheck",
    "NoTechSupport",
    "NoOnlineSecurity",
    "ServiceRiskIndicator",
    "HighRiskCustomer"
]

df[new_features].head()

,TenureGroup,IsNewCustomer,IsLongTermCustomer,NumberOfServices,OptionalServiceCount,HasInternetService,IsFiberCustomer,MonthlyChargeGroup,TotalChargeGroup,ChargePerTenure,HighValueCustomer,MonthToMonthContract,ElectronicCheck,NoTechSupport,NoOnlineSecurity,ServiceRiskIndicator,HighRiskCustomer
0,New,1,0,3,1,1,0,Low,Low,29.850000,0,1,1,1,1,1,0
1,Mid,0,0,4,2,1,0,Medium,High,55.573529,0,0,0,1,0,0,0
2,New,1,0,4,2,1,0,Medium,Low,54.075000,0,1,0,1,0,0,0
3,Mid,0,0,5,3,1,0,Medium,High,40.905556,0,0,0,0,0,0,0
4,New,1,0,2,0,1,1,High,Low,75.825000,0,1,1,1,1,1,0


In [24]:
# Analyze engineered features

tenure_churn = pd.crosstab(
    df["TenureGroup"],
    df["Churn"],
    normalize="index"
) * 100

tenure_churn

Churn,No,Yes
TenureGroup,,
New,52.561757,47.438243
Early,71.289062,28.710938
Mid,79.611041,20.388959
Long-term,90.486824,9.513176


In [25]:
df["Churn"] = df["Churn"].map({
    "No": 0,
    "Yes": 1
})

In [26]:

tenure_churn = pd.crosstab(
    df["TenureGroup"],
    df["Churn"],
    normalize="index"
) * 100

tenure_churn

Churn,0,1
TenureGroup,,
New,52.561757,47.438243
Early,71.289062,28.710938
Mid,79.611041,20.388959
Long-term,90.486824,9.513176


In [27]:
# Churn by number of services

service_churn = (
    df.groupby("NumberOfServices")["Churn"]
    .mean()
    .mul(100)
    .reset_index()
)

service_churn

,NumberOfServices,Churn
0,1,8.699324
1,2,30.666667
2,3,47.671569
3,4,37.722048
4,5,31.288344
5,6,25.723473
6,7,21.745152
7,8,11.666667
8,9,5.288462


In [28]:
# Churn by monthly charge group

charge_churn = (
    df.groupby("MonthlyChargeGroup", observed=True)["Churn"]
    .mean()
    .mul(100)
    .reset_index()
)

charge_churn

,MonthlyChargeGroup,Churn
0,Low,11.237230
1,Medium,24.575311
2,High,37.507114
3,Very High,32.878271


In [29]:
# Churn by high-value status

high_value_churn = (
    df.groupby("HighValueCustomer")["Churn"]
    .mean()
    .mul(100)
    .reset_index()
)

high_value_churn

,HighValueCustomer,Churn
0,0,24.449924
1,1,32.749859


In [30]:
# Churn by risk indicator

risk_churn = (
    df.groupby("HighRiskCustomer")["Churn"]
    .mean()
    .mul(100)
    .reset_index()
)

risk_churn

,HighRiskCustomer,Churn
0,0,24.959719
1,1,76.388889


In [31]:
# Check for missing values

print(
    df.isnull().sum()[
        df.isnull().sum() > 0
    ]
)

Series([], dtype: int64)


In [32]:
print("Duplicate rows:", df.duplicated().sum())

Duplicate rows: 0


In [33]:
print("Final shape:", df.shape)

print("\nFinal columns:")
for column in df.columns:
    print(column)

Final shape: (7043, 38)

Final columns:
customerID
gender
SeniorCitizen
Partner
Dependents
tenure
PhoneService
MultipleLines
InternetService
OnlineSecurity
OnlineBackup
DeviceProtection
TechSupport
StreamingTV
StreamingMovies
Contract
PaperlessBilling
PaymentMethod
MonthlyCharges
TotalCharges
Churn
TenureGroup
IsNewCustomer
IsLongTermCustomer
NumberOfServices
OptionalServiceCount
HasInternetService
IsFiberCustomer
MonthlyChargeGroup
TotalChargeGroup
ChargePerTenure
HighValueCustomer
MonthToMonthContract
ElectronicCheck
NoTechSupport
NoOnlineSecurity
ServiceRiskIndicator
HighRiskCustomer


In [34]:
# Save feature-engineered dataset

PROCESSED_DIR = BASE_DIR / "data" / "processed"

OUTPUT_PATH = (
    PROCESSED_DIR
    / "feature_engineered_data.csv"
)

df.to_csv(
    OUTPUT_PATH,
    index=False
)

print("Feature-engineered dataset saved:")
print(OUTPUT_PATH)

Feature-engineered dataset saved:
/home/aximsoft/Downloads/Weekend_Task/AI Customer Intelligence Platform /data/processed/feature_engineered_data.csv
